# 47. YOLO26n-pose를 ONNX로 바꾸면 관절 좌표와 체간 기울기가 얼마나 달라질까?

검증 선수 6명에게서 이미 판독한 이륙과 무릎 통과 12장을 사용합니다. 추론 엔진만 바꾸고 입력 픽셀과 사람 선택 기준을 맞춥니다. 입력은 긴 변 640, 실제 형태는 1 × 3 × 640 × 384입니다. 사람 점수 0.25, 관절 점수 0.50을 사용합니다.

Mac 프로젝트의 Python 커널로 위에서부터 실행하세요. ONNX 패키지가 없는 환경에서는 프로젝트 폴더에서 `uv pip install onnx onnxruntime`을 먼저 실행합니다. `models/yolo26n-pose.pt`와 원본 영상이 필요합니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from wlcv.evaluation.onnx_pose import compare, overlay

folder = root / "outputs/onnx_pose_parity" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
frames, joints, changes = compare(folder)
print(folder)

## 가까운 쪽 네 관절

파란 원은 PyTorch, 자홍색 십자는 ONNX입니다. 노란 ×는 사람이 직접 본 점, 삼각형은 추정점입니다. 아래 `person`을 바꿔 다른 선수도 확인할 수 있습니다. b02 30번은 23번 실험에서 확인한 두 번째 사람 박스를 사용합니다. 대응이 모호하면 결과표에 빈칸이 남습니다.

In [ ]:
person = "b02"  # a03, a05, b02, b03, c04, c08
fig = overlay(person, frames, joints)
fig.savefig(folder / f"{person}_overlay.png", dpi=140)
plt.show()

paired = joints.pt_status.eq("표시") & joints.ort_status.eq("표시")
summary = joints.assign(paired=paired, shown_px=joints.distance_px.where(paired)).groupby("joint", sort=False).agg(
    total=("frame", "size"), paired=("paired", "sum"), median_px=("shown_px", "median"),
    max_px=("shown_px", "max"), max_score_diff=("score_diff", "max"),
    manual_n=("pt_manual_px", "count"), pt_manual_mae_px=("pt_manual_px", "mean"),
    ort_manual_mae_px=("ort_manual_px", "mean"))
display(summary.round(7))
print("관절 상태 불일치", joints.pt_status.ne(joints.ort_status).sum(), "개")

`paired`는 두 런타임 모두 표시 기준을 충족한 점의 수입니다. `max_px`는 같은 관절 좌표 간 유클리드 거리의 최댓값입니다. 마지막 두 열은 직접 확인한 수동점까지의 평균 거리(px)입니다. 추정·판독 불가·라벨 없음은 이 평균에서 제외합니다.

## 시점별 체간 기울기와 두 시점의 변화량

골반에서 어깨를 잇는 선이 화면의 위쪽 수직선과 이루는 각도를 씁니다. 무릎 통과에서 이륙 값을 뺀 변화량이 음수이면 체간이 더 수직에 가까워졌습니다. 어깨와 골반 둘 다 기준을 통과한 프레임만 계산합니다.

In [ ]:
display(frames[["person", "phase", "frame", "side", "pt_id", "ort_id", "performer_iou",
                "pt_angle_deg", "ort_angle_deg", "angle_diff_deg"]].round(7))
display(changes.round(7))
print("직접 수동점으로 각도를 확인할 수 있는 장수", frames.manual_angle_deg.count())

checks = pd.Series({
    "수행자 12장 대응": frames[["pt_id", "ort_id"]].notna().all().all() and len(frames) == 12,
    "사람 후보 수 일치": frames.pt_people.eq(frames.ort_people).all(),
    "관절 상태 일치": joints.pt_status.eq(joints.ort_status).all(),
    "표시된 관절 좌표 거리 ≤ 0.10 px": paired.any() and joints.loc[paired, "distance_px"].le(0.10).all(),
    "수행자 관절 신뢰도 차이 ≤ 0.0001": joints.score_diff.le(0.0001).all(),
    "12장 각도 차이 ≤ 0.01°": frames.angle_diff_deg.le(0.01).all(),
    "6명 변화량 차이 ≤ 0.01°": changes.diff_deg.le(0.01).all(),
    "6명 변화 방향 일치": np.sign(changes.pt_change_deg).eq(np.sign(changes.ort_change_deg)).all(),
})
checks.to_csv(folder / "checks.csv", header=["passed"])
display(checks)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4), layout="constrained")
ax.axhline(0, color="0.7", lw=1)
ax.plot(changes.index, changes.pt_change_deg, "o", fillstyle="none", ms=10, label="PyTorch")
ax.plot(changes.index, changes.ort_change_deg, "+", ms=9, label="ONNX Runtime")
ax.set(ylabel="Knee pass minus lift-off (degrees)", xlabel="Validation athlete")
ax.legend()
fig.savefig(folder / "angle_changes.png", dpi=140)
plt.show()

원시 좌표와 관절 신뢰도는 실행 폴더의 `joints.csv`, 프레임별 수행자 대응과 각도는 `frames.csv`, 선수별 변화는 `changes.csv`에 남습니다. `pose.onnx`를 로컬 Netron에서 열면 입력과 출력 채널을 확인할 수 있습니다. 이 실험의 허용 차이는 좌표 거리 0.10 px, 신뢰도 0.0001, 각도 및 변화량 0.01°입니다. 표시 상태와 수행자 대응도 함께 확인합니다.